# Foundational: Dataset-Level ML Lineage with DVC and MLflow on Amazon SageMaker AI

This notebook demonstrates how to build traceable ML workflows using [DVC](https://dvc.org/) for data versioning and [fully managed MLflow on Amazon SageMaker AI](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html) for experiment tracking. You'll learn how to:

- Version image datasets with DVC and store them in Amazon S3
- Track experiments and link models to specific data versions with MLflow
- Run SageMaker AI Processing and Training jobs with full data lineage
- Deploy models using the SageMaker v3 SDK (ModelTrainer → ModelBuilder)

This enables **dataset-level lineage** — every model links to a DVC commit hash that points to the exact processed dataset in S3. Given any model, you can reconstruct its training data with `dvc pull`. However, DVC versions the dataset as a unit — there's no structured metadata about which individual records are inside each version. To find out, you'd need to pull the dataset and inspect its contents.

If you need to query which specific records trained a model without reconstructing the dataset (e.g., for opt-out requests or audits), see the [healthcare-compliance](../healthcare-compliance/) variant, which adds a record-level manifest.

### DVC (Data Version Control)

[DVC](https://dvc.org/) extends Git to handle large files and datasets. Instead of storing data in Git, DVC tracks lightweight `.dvc` metafiles while the actual data lives in remote storage like S3.

### MLflow on Amazon SageMaker AI

[MLflow on Amazon SageMaker AI](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html) makes it easier to track experiments and monitor performance of models and AI applications using a single tool. SageMaker AI provides a fully managed MLflow App that integrates natively with SageMaker AI Training jobs.

---

### CIFAR-10 Dataset

We use the [CIFAR-10 dataset](https://www.cs.toronto.edu/~kriz/cifar.html), which contains 60,000 32x32 color images in 10 classes:
airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck

### Data Versioning Story: More Data = Better Model

We simulate a common real-world scenario where you start with limited labeled data and later expand your dataset:

- **v1.0**: Train on 5% of the data (~2,250 training images)
- **v2.0**: Train on 10% of the data (~4,500 training images)

This demonstrates how DVC tracks different dataset versions and MLflow lets you compare model performance as your data grows. You can adjust these fractions to use more data for better accuracy.

## Prerequisites
---

This notebook can run in Amazon SageMaker Studio, a SageMaker AI notebook instance, or locally with AWS credentials configured.

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠️ Compatibility Notice:</strong>This notebook has been tested using <strong>SageMaker Python SDK version 3.4.1</strong>.
<br> with the following Python runtime versions:
<ul>
<li><strong>Python 3.11</strong></li>
<li><strong>Python 3.12</strong></li>
</ul>
</div>

In [ ]:
# Install dependencies
%pip install --no-cache-dir -r ../requirements.txt

### Restart Your Kernel

In [ ]:
# Restart kernel to get the packages
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
import sagemaker
import mlflow
import torch
from importlib.metadata import version
import sys

py_version = f"py{sys.version_info.major}{sys.version_info.minor}"
print(py_version)

PYTORCH_FRAMEWORK_VERSION = {
    "py311": "2.5",
    "py312": "2.6",
}

if py_version not in PYTORCH_FRAMEWORK_VERSION:
    raise RuntimeError(f"No PyTorch inference container for {py_version}. Supported: py311, py312")

pytorch_framework_version = PYTORCH_FRAMEWORK_VERSION[py_version]

print(f"SageMaker SDK version: {version('sagemaker')}")
print(f"MLflow version: {mlflow.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"Python version: {py_version}")
print(f"Pytorch framework version: {pytorch_framework_version}")

## Part 1: Configure DVC for Data Versioning
---

We create a subdirectory with a git repository to store DVC metadata. The actual data is stored in Amazon S3.

**Note:** This example uses AWS CodeCommit, but DVC works with any Git provider (GitHub, GitLab, Bitbucket, etc.). Simply replace the `git remote add origin` URL with your repository URL and configure appropriate credentials. The key requirement is that your SageMaker AI execution role (or notebook IAM role) must have permissions to access the Git repository — for CodeCommit, this means `codecommit:GitPull` and `codecommit:GitPush` permissions.

In [ ]:
# Define the DVC repository name
dvc_repo_name = "cifar10-dvc-demo-11"

In [ ]:
%%bash -s "$dvc_repo_name"

repo_name="$1"

# Create CodeCommit repository
aws codecommit create-repository --repository-name ${repo_name} \
    --repository-description "CIFAR-10 image classification with DVC versioning"

account=$(aws sts get-caller-identity --query Account --output text)
region=$(python -c "import boto3;print(boto3.Session().region_name)")
region=${region:-us-east-1}

mkdir -p ${repo_name}
cd ${repo_name}

# Initialize git repo
git init
git branch -M main  
git remote add origin codecommit::${region}://${repo_name}

# Configure git
git config --global user.email "user@domain.com"
git config --global user.name "user"
git config --global credential.helper '!aws codecommit credential-helper $@'
git config --global credential.UseHttpPath true

# Initialize DVC
dvc init
git commit -m 'Initialize DVC'

# Set DVC remote to S3
dvc remote add -d storage s3://sagemaker-${region}-${account}/DEMO-cifar10-dvc
git commit .dvc/config -m "Configure DVC remote"

# Set DVC cache to S3
dvc remote add s3cache s3://sagemaker-${region}-${account}/DEMO-cifar10-dvc/cache
dvc config cache.s3 s3cache
dvc config core.analytics false

git add .dvc/config
git commit -m 'Update DVC config'

git push --set-upstream origin main

## Part 2: Processing and Training with DVC and SageMaker AI
---

We'll run two experiments to show how model performance improves with more data:

- **Experiment 1 (v1.0)**: Train with 5% of the data
- **Experiment 2 (v2.0)**: Train with 10% of the data

Both experiments version the data with DVC and log metrics to MLflow for comparison.

In [ ]:
import boto3
import json
from datetime import datetime

from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core.image_uris import get_training_image_uri
from sagemaker.core.processing import FrameworkProcessor
from sagemaker.core.shapes import ProcessingOutput, ProcessingS3Output
from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.train.configs import SourceCode, Compute
from sagemaker.core import image_uris

# Setup session
sess = Session()
role = get_execution_role()
region = sess.boto_region_name
bucket = sess.default_bucket()
account = boto3.client('sts').get_caller_identity()['Account']

dvc_repo_url = f"codecommit::{region}://{dvc_repo_name}"
prefix = 'DEMO-cifar10-dvc'

print(f"Account: {account}")
print(f"Bucket: {bucket}")
print(f"Region: {region}")
print(f"Role: {role}")

### Setup MLflow Tracking

Create or connect to a SageMaker AI MLflow App for experiment tracking.

> **Estimated time:** Creating a new MLflow App takes ~3-5 minutes.

**Note:** The following cells create an IAM role and MLflow App programmatically. Your notebook's IAM role must have `iam:CreateRole` and `iam:PutRolePolicy` permissions. 

Alternatively, you can create the MLflow App via the [Amazon SageMaker AI console](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow-create-tracking-server-studio.html) and skip the role creation cell — just update `mlflow_app_name` to match your existing app.

In [ ]:
import mlflow
import time

sm_client = boto3.client('sagemaker')
iam = boto3.client('iam')

experiment_name = 'demo-cifar10-mlflow-dvc'
mlflow_app_name = 'cifar10-mlflow-app'
mlflow_role_name = 'MLflowAppIAMRole'

# Create IAM role for MLflow if needed
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "sagemaker.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
}

# Least-privilege policy for MLflow App role
# Based on AWS docs: https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow-app-setup-prerequisites-iam.html
# S3 actions scoped to SageMaker bucket only
mlflow_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "s3:Get*",
                "s3:Put*",
                "s3:List*"
            ],
            "Resource": [
                f"arn:aws:s3:::{bucket}",
                f"arn:aws:s3:::{bucket}/*"
            ]
        },
        {
            "Effect": "Allow",
            "Action": [
                "sagemaker:AddTags",
                "sagemaker:CreateModelPackageGroup",
                "sagemaker:CreateModelPackage",
                "sagemaker:UpdateModelPackage",
                "sagemaker:DescribeModelPackageGroup"
            ],
            "Resource": "*"
        }
    ]
}

try:
    iam.get_role(RoleName=mlflow_role_name)
    print(f"Using existing role: {mlflow_role_name}")
except iam.exceptions.NoSuchEntityException:
    print(f"Creating IAM role: {mlflow_role_name}")
    iam.create_role(
        RoleName=mlflow_role_name,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description='IAM role for MLflow App'
    )
    iam.put_role_policy(
        RoleName=mlflow_role_name,
        PolicyName='MLflowAppAccess',
        PolicyDocument=json.dumps(mlflow_policy)
    )
    # Wait for IAM role to propagate before using it
    print("Waiting for IAM role to propagate...")
    time.sleep(10)

mlflow_role_arn = f"arn:aws:iam::{account}:role/{mlflow_role_name}"
print(f"MLflow Role ARN: {mlflow_role_arn}")

In [ ]:
# Check for existing MLflow App by name, or create new one
apps = sm_client.list_mlflow_apps().get('Summaries', [])
mlflow_app = next((a for a in apps if a['Name'] == mlflow_app_name), None)

if mlflow_app:
    print(f"Using existing MLflow App: {mlflow_app['Name']}")
    mlflow_app = sm_client.describe_mlflow_app(Arn=mlflow_app['Arn'])
else:
    print(f"Creating MLflow App: {mlflow_app_name}...")
    response = sm_client.create_mlflow_app(
        Name=mlflow_app_name,
        ArtifactStoreUri=f's3://{bucket}',
        RoleArn=mlflow_role_arn
    )
    while True:
        mlflow_app = sm_client.describe_mlflow_app(Arn=response['Arn'])
        if mlflow_app['Status'] == 'Created':
            break
        elif mlflow_app['Status'] in ['CreateFailed', 'Deleted']:
            raise RuntimeError(f"MLflow App creation failed: {mlflow_app['Status']}")
        print(f"Status: {mlflow_app['Status']}... waiting")
        time.sleep(30)

mlflow_app_arn = mlflow_app['Arn']
print(f"MLflow App ARN: {mlflow_app_arn}")

### Experiment 1: Train with 5% of Data (v1.0)
---

First, we process CIFAR-10 using only 5% of the training data. This simulates having limited labeled data when starting a project.

In [ ]:
# Define version for experiment 1
data_version_v1 = "v1.0"
data_fraction_v1 = 0.05  # 5% of training data

timestamp = datetime.now().strftime("%m-%d-%y_%H%M")  # 01-28-26_1427
pipeline_run_id_v1 = f"{data_version_v1}-{timestamp}"

print(f"Pipeline run ID: {pipeline_run_id_v1}")
print(f"Data fraction: {data_fraction_v1} ({int(data_fraction_v1 * 100)}% of training data)")

#### Run Processing Job (v1.0)

> **Estimated time:** ~4-5 minutes

In [ ]:
# Get PyTorch image for processing
processing_image = get_training_image_uri(
    region=region,
    framework="pytorch",
    framework_version=pytorch_framework_version,
    py_version=py_version,
    instance_type="ml.m5.xlarge",
)

processor_v1 = FrameworkProcessor(
    image_uri=processing_image,
    role=role,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    env={
        "DVC_REPO_URL": dvc_repo_url,
        "DVC_REPO_NAME": dvc_repo_name,
        "MLFLOW_TRACKING_URI": mlflow_app_arn,
        "MLFLOW_EXPERIMENT_NAME": experiment_name,
        "PIPELINE_RUN_ID": pipeline_run_id_v1,
    }
)

print(f"Processing image: {processing_image}")

In [ ]:
%%time

processor_v1.run(
    code="preprocessing_foundational.py",
    source_dir="../source_dir",
    arguments=[
        "--data-fraction", str(data_fraction_v1),
        "--data-version", data_version_v1,
        "--val-split", "0.1"
    ],
    wait=True
)

#### Run Training Job (v1.0)

> **Estimated time:** ~4-5 minutes. Includes instance provisioning, DVC pull of the versioned dataset, and training MobileNetV3-Small for 5 epochs.

In [ ]:
# Get PyTorch training image
training_image = image_uris.retrieve(
    framework="pytorch",
    region=region,
    version=pytorch_framework_version,
    py_version=py_version,
    instance_type="ml.m5.xlarge",
    image_scope="training"
)

print(f"Training image: {training_image}")

In [ ]:
# Common model name for both versions (same architecture)
registered_model_name = "CIFAR10-MobileNetV3"

model_trainer_v1 = ModelTrainer(
    sagemaker_session=sess,
    training_image=training_image,
    source_code=SourceCode(
        source_dir="../source_dir",
        entry_script="train.py",
        requirements="requirements.txt",
    ),
    compute=Compute(
        instance_type="ml.m5.xlarge",
        instance_count=1,
        volume_size_in_gb=30,
    ),
    base_job_name="cifar10-mobilenet-v1",
    hyperparameters={
        "epochs": 5,
        "batch_size": 256,
        "learning_rate": 0.001,
    },
    environment={
        "DVC_REPO_URL": dvc_repo_url,
        "DVC_REPO_NAME": dvc_repo_name,
        "DATA_VERSION": data_version_v1,
        "PIPELINE_RUN_ID": pipeline_run_id_v1,
        "MLFLOW_TRACKING_URI": mlflow_app_arn,
        "MLFLOW_EXPERIMENT_NAME": experiment_name,
        "MLFLOW_REGISTERED_MODEL_NAME": registered_model_name,
    }
)

In [ ]:
%%time

model_trainer_v1.train()

### Experiment 2: Train with 10% of Data (v2.0)
---

Now we double the dataset size. This simulates having collected more labeled data over time. We expect to see improved accuracy compared to v1.0.

> **Estimated time:** ~8-10 minutes total for processing and training.

In [ ]:
# Define version for experiment 2
data_version_v2 = "v2.0"
data_fraction_v2 = 0.1  # 10% of training data

timestamp = datetime.now().strftime("%m-%d-%y_%H%M")  # 01-28-26_1427
pipeline_run_id_v2 = f"{data_version_v2}-{timestamp}"

print(f"Pipeline run ID: {pipeline_run_id_v2}")
print(f"Data fraction: {data_fraction_v2} ({int(data_fraction_v2 * 100)}% of training data)")

In [ ]:
processor_v2 = FrameworkProcessor(
    image_uri=processing_image,
    role=role,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    env={
        "DVC_REPO_URL": dvc_repo_url,
        "DVC_REPO_NAME": dvc_repo_name,
        "MLFLOW_TRACKING_URI": mlflow_app_arn,
        "MLFLOW_EXPERIMENT_NAME": experiment_name,
        "PIPELINE_RUN_ID": pipeline_run_id_v2,
    }
)

In [ ]:
%%time

processor_v2.run(
    code="preprocessing_foundational.py",
    source_dir="../source_dir",
    arguments=[
        "--data-fraction", str(data_fraction_v2),
        "--data-version", data_version_v2,
        "--val-split", "0.1"
    ],
    wait=True
)

In [ ]:
model_trainer_v2 = ModelTrainer(
    sagemaker_session=sess,
    training_image=training_image,
    source_code=SourceCode(
        source_dir="../source_dir",
        entry_script="train.py",
        requirements="requirements.txt",
    ),
    compute=Compute(
        instance_type="ml.m5.xlarge",
        instance_count=1,
        volume_size_in_gb=30,
    ),
    base_job_name="cifar10-mobilenet-v2",
    hyperparameters={
        "epochs": 5,
        "batch_size": 256,
        "learning_rate": 0.001,
    },
    environment={
        "DVC_REPO_URL": dvc_repo_url,
        "DVC_REPO_NAME": dvc_repo_name,
        "DATA_VERSION": data_version_v2,
        "PIPELINE_RUN_ID": pipeline_run_id_v2,
        "MLFLOW_TRACKING_URI": mlflow_app_arn,
        "MLFLOW_EXPERIMENT_NAME": experiment_name,
        "MLFLOW_REGISTERED_MODEL_NAME": registered_model_name,
    }
)

In [ ]:
%%time

model_trainer_v2.train()

### Compare Experiments in MLflow
---

Now you can compare the two experiments in the MLflow UI. To access the UI, see [Launch the MLflow UI using a presigned URL](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow-launch-ui.html).

You should see:
- v1.0 (5% data): Lower accuracy
- v2.0 (10% data): Higher accuracy

Each run is linked to its exact data version via the `data_version` and `data_git_commit_id` parameters.

![MLflow Experiment Comparison](../img/mlflow_experiment.png)

### Training Run Details
---

Click into any run to see detailed metrics, parameters, and artifacts. Key information includes:
- Training/validation loss curves over epochs
- Hyperparameters used (learning rate, batch size, epochs)
- DVC data version and Git commit linking the exact dataset

![MLflow Training Run Details](../img/mlflow_training_run.png)

### Registered Model
---

Models are automatically registered in the MLflow Model Registry. This provides:
- Version history of all trained models
- Stage transitions (Staging → Production)
- Direct links to the training run and data version that produced each model

![MLflow Registered Model](../img/mlflow_registered_model.png)

## Part 3: Deploy Model with ModelBuilder
---

Deploy the best model (v2.0, trained on more data) from MLflow registry to a SageMaker AI endpoint.

> **Estimated time:** ~4-5 minutes for endpoint deployment.

In [ ]:
from mlflow import MlflowClient
from sagemaker.serve.model_builder import ModelBuilder
from sagemaker.serve.builder.schema_builder import SchemaBuilder
from sagemaker.serve.mode.function_pointers import Mode

# Connect to MLflow
mlflow.set_tracking_uri(mlflow_app_arn)
client = MlflowClient()

# Get the latest model version
registered_model_name = "CIFAR10-MobileNetV3"
registered_model = client.get_registered_model(name=registered_model_name)
latest_version = registered_model.latest_versions[0]

model_version = latest_version.version
model_source = latest_version.source
mlflow_model_path = f"models:/{registered_model_name}/{model_version}"

print(f"Model: {registered_model_name}")
print(f"Version: {model_version}")
print(f"Source: {model_source}")

In [ ]:
import torch
import json
import io
from PIL import Image
from torchvision import transforms
from sagemaker.serve.marshalling.custom_payload_translator import CustomPayloadTranslator

class ImageInputTranslator(CustomPayloadTranslator):
    """Handles raw image bytes (JPEG/PNG)"""
    def __init__(self):
        super().__init__(content_type='image/jpeg', accept_type='application/json')
        # MobileNetV3 with 96x96 input
        self.transform = transforms.Compose([
            transforms.Resize((96, 96)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    
    def serialize_payload_to_bytes(self, payload: object) -> bytes:
        # Client-side: send raw image bytes
        if isinstance(payload, bytes):
            return payload
        raise ValueError('Expected bytes')
    
    def deserialize_payload_from_stream(self, stream) -> object:
        # Server-side: decode image and transform to tensor
        image_bytes = stream.read()
        image = Image.open(io.BytesIO(image_bytes)).convert('RGB')
        tensor = self.transform(image).unsqueeze(0)
        # Move to GPU if available (matches model device)
        if torch.cuda.is_available():
            tensor = tensor.cuda()
        return tensor

class ImageOutputTranslator(CustomPayloadTranslator):
    """Converts model output tensor to JSON."""
    def __init__(self):
        super().__init__(content_type='application/json', accept_type='application/json')
    
    def serialize_payload_to_bytes(self, payload: object) -> bytes:
        if isinstance(payload, torch.Tensor):
            return json.dumps(payload.tolist()).encode('utf-8')
        return json.dumps(payload).encode('utf-8')
    
    def deserialize_payload_from_stream(self, stream) -> object:
        return json.load(stream)

# Sample input: raw JPEG bytes (we'll create a tiny valid JPEG)
# For schema, we just need representative samples
sample_image = Image.new('RGB', (32, 32), color='red')
buffer = io.BytesIO()
sample_image.save(buffer, format='JPEG')
sample_input = buffer.getvalue()

# Sample output: class probabilities for 10 CIFAR-10 classes
sample_output = [[0.1] * 10]

schema_builder = SchemaBuilder(
    sample_input=sample_input,
    sample_output=sample_output,
    input_translator=ImageInputTranslator(),
    output_translator=ImageOutputTranslator()
)

In [ ]:
from sagemaker.core import image_uris

inference_image = image_uris.retrieve(
    framework="pytorch",
    region=region,
    version=pytorch_framework_version,
    py_version=py_version,
    instance_type="ml.m5.xlarge",
    image_scope="inference"
)

In [ ]:
model_builder = ModelBuilder(
    mode=Mode.SAGEMAKER_ENDPOINT,
    image_uri=inference_image,
    instance_type="ml.m5.xlarge",
    schema_builder=schema_builder,
    model_metadata={
        "MLFLOW_MODEL_PATH": mlflow_model_path,
        "MLFLOW_TRACKING_ARN": mlflow_app_arn
    },
    dependencies={"auto": False, "custom": [
        "mlflow==3.4.0",
        "sagemaker-mlflow>=0.2.0",
        "sagemaker==3.4.0",
        "cloudpickle==3.1.2",
        "numpy==2.4.1"
    ]},
)

print(f"ModelBuilder configured with: {mlflow_model_path}")

In [ ]:
import uuid

unique_id = str(uuid.uuid4())[:8]
model_name = f"cifar10-mobilenet-{unique_id}"
endpoint_name = f"cifar10-endpoint-{unique_id}"

# Build the model
core_model = model_builder.build(model_name=model_name, region=region)
print(f"Model built: {core_model.model_name}")

In [ ]:
# Deploy to endpoint
core_endpoint = model_builder.deploy(
    endpoint_name=endpoint_name,
    initial_instance_count=1
)

print(f"Endpoint deployed: {core_endpoint.endpoint_name}")

### Test the Endpoint

In [ ]:
import json
import numpy as np
import requests
from io import BytesIO
from PIL import Image


# Download a test image (dog from PyTorch examples)
url = 'https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg'
response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
response.raise_for_status()
image_bytes = response.content

img = Image.open(BytesIO(image_bytes))
img.thumbnail((300, 300))
img


In [ ]:
runtime_client = boto3.client('sagemaker-runtime')
response = runtime_client.invoke_endpoint(
    EndpointName=core_endpoint.endpoint_name,
    Body=image_bytes,
    ContentType='image/jpeg'
)
prediction = json.loads(response['Body'].read().decode('utf-8'))

# Results
predicted_class = np.argmax(prediction)
confidence = prediction[0][predicted_class]

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print(f'Predicted: {class_names[predicted_class]}')
print(f'All probabilities:')
for name, prob in zip(class_names, prediction[0]):
    print(f'  {name}: {prob:.3f}')

## Cleanup
---

Delete resources to avoid ongoing charges.

In [ ]:
# Delete endpoint
from sagemaker.core.resources import EndpointConfig

core_endpoint_config = EndpointConfig.get(endpoint_config_name=core_endpoint.endpoint_name)
core_model.delete()
core_endpoint.delete()
core_endpoint_config.delete()

print("Endpoint resources deleted!")

In [ ]:
# Optional: Delete MLflow App
# sm_client.delete_mlflow_app(Arn=mlflow_app_arn)

In [ ]:
# Optional: Delete CodeCommit repository
# %aws codecommit delete-repository --repository-name {dvc_repo_name}

## Going Further
---

This demo versions data at the dataset level. Here are some directions to extend it:

- **Increase data fractions** — Try training with 25%, 50%, or 100% of CIFAR-10 (adjust `data_fraction`) and compare accuracy improvements in MLflow across more data versions
- **Reproducibility** — Given any model in MLflow, extract its `data_git_commit_id`, run `git checkout <tag> && dvc pull` to recreate the exact training data. This is the core value of dataset-level lineage: any model can be reproduced from its DVC commit
- **Record-level lineage** — If you need to trace individual records (e.g., "which models used record X's data?"), see the [healthcare-compliance](../healthcare-compliance/) variant, which adds a record-level manifest and audit queries

### Speeding Up Iteration

When running repeated experiments (like the v1.0 → v2.0 flow above), two SageMaker AI features help streamline the process:

- **[SageMaker AI Managed Warm Pools](https://docs.aws.amazon.com/sagemaker/latest/dg/train-warm-pools.html)** — Keep training instances warm between jobs so back-to-back training runs reuse already-provisioned infrastructure. Add `keep_alive_period_in_seconds` to your `Compute` config to enable it. Note that warm pools apply to training jobs only, not processing jobs.

- **[SageMaker AI Pipelines](https://docs.aws.amazon.com/sagemaker/latest/dg/pipelines-overview.html)** — Orchestrate the processing → training → registration workflow as a single, repeatable pipeline instead of running each step manually in a notebook. Pipelines handle step dependencies, pass artifacts between steps automatically, and can be triggered programmatically (e.g., when new data is available and you want to create a new dataset version).